In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random

np.random.seed(42)
random.seed(42)

# -----------------------------
# 1. Generate employees.csv
# -----------------------------

roles_config = [
    ("Loan Officer", 25),
    ("Underwriter", 15),
    ("Processor", 15),
    ("Account Executive", 10),
    ("Branch Manager", 5),
    ("Regional Manager", 3),
    ("Operations Director", 2),
]

regions = ["East", "West", "Midwest", "South"]
branches = [
    (12, "Charlotte", "East"),
    (18, "Los Angeles", "West"),
    (22, "Chicago", "Midwest"),
    (30, "Atlanta", "South"),
    (35, "Dallas", "South"),
]

first_names = [
    "Emily", "Marcus", "Sophia", "Daniel", "Ashley", "Michael", "Rebecca",
    "Jonathan", "Olivia", "Liam", "Noah", "Ava", "Isabella", "Ethan",
    "Mason", "Logan", "Lucas", "Mia", "Amelia", "Harper"
]
last_names = [
    "Carson", "Lee", "Ramirez", "Nguyen", "Patel", "Turner", "Shaw",
    "Price", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller",
    "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez"
]

employees = []
employee_id_counter = 100

for role, count in roles_config:
    for _ in range(count):
        first = random.choice(first_names)
        last = random.choice(last_names)
        if role in ["Branch Manager", "Regional Manager", "Operations Director"]:
            region = random.choice(regions)
            branch_id, branch_name, branch_region = random.choice(branches)
            region = branch_region
        else:
            branch_id, branch_name, region = random.choice(branches)

        hire_year = random.randint(2012, 2022)
        hire_month = random.randint(1, 12)
        hire_day = random.randint(1, 28)
        hire_date = datetime(hire_year, hire_month, hire_day).date()

        status = "Active"

        if role in ["Loan Officer", "Account Executive"]:
            comp_plan_type = "bps"
            base_salary = None
            target_bps = random.choice([80, 90, 100, 110, 120, 130])
        else:
            comp_plan_type = "salary"
            base_salary = random.choice([52000, 65000, 78000, 95000, 115000, 150000])
            target_bps = None

        employees.append({
            "employee_id": employee_id_counter,
            "first_name": first,
            "last_name": last,
            "role": role,
            "region": region,
            "branch_id": branch_id,
            "branch_name": branch_name,
            "hire_date": hire_date,
            "status": status,
            "comp_plan_type": comp_plan_type,
            "base_salary": base_salary,
            "target_bps": target_bps,
            "manager_id": None  # you can enhance this later if you want hierarchy
        })

        employee_id_counter += 1

employees_df = pd.DataFrame(employees)

# Save employees.csv
employees_df.to_csv("employees.csv", index=False)

# Helper: get IDs by role
def get_employee_ids_by_role(role):
    return employees_df.loc[employees_df["role"] == role, "employee_id"].tolist()

loan_officers = get_employee_ids_by_role("Loan Officer")
underwriters = get_employee_ids_by_role("Underwriter")
processors = get_employee_ids_by_role("Processor")
account_execs = get_employee_ids_by_role("Account Executive")

# -----------------------------
# 2. Generate loans.csv
# -----------------------------

num_loans = 10000

states = ["NC", "SC", "GA", "FL", "CA", "NY", "IL", "TX", "VA", "TN"]
property_types = ["SFR", "Condo", "2-4 Unit"]
occupancies = ["Primary", "Second Home", "Investment"]
purposes = ["Purchase", "Refi RT", "Refi CO"]
channels = ["Retail", "Wholesale"]
products = ["Conventional", "FHA", "VA", "Jumbo"]

loan_statuses = ["Funded", "Withdrawn", "Denied", "Suspended", "Canceled"]

base_date = datetime(2023, 1, 1)

loans = []

for i in range(num_loans):
    loan_number = 70000001 + i

    app_offset = random.randint(0, 90)
    application_date = base_date + timedelta(days=app_offset)

    lock_offset = random.randint(0, 5)
    lock_date = application_date + timedelta(days=lock_offset)

    property_state = random.choices(
        states,
        weights=[10, 5, 7, 8, 12, 10, 8, 10, 5, 5],
        k=1
    )[0]

    if property_state in ["CA", "NY"]:
        loan_amount = int(np.random.normal(650000, 120000))
    elif property_state in ["FL", "IL", "TX", "VA"]:
        loan_amount = int(np.random.normal(400000, 80000))
    else:
        loan_amount = int(np.random.normal(275000, 60000))

    loan_amount = max(80000, min(loan_amount, 1200000))

    property_type = random.choices(property_types, weights=[70, 20, 10], k=1)[0]
    occupancy_type = random.choices(occupancies, weights=[75, 10, 15], k=1)[0]
    loan_purpose = random.choices(purposes, weights=[55, 30, 15], k=1)[0]
    channel = random.choices(channels, weights=[60, 40], k=1)[0]

    if loan_amount > 726200:
        product_type = "Jumbo"
    else:
        product_type = random.choices(products, weights=[60, 20, 10, 10], k=1)[0]

    if product_type == "FHA":
        rate = round(np.random.normal(5.75, 0.25), 3)
    elif product_type == "VA":
        rate = round(np.random.normal(5.875, 0.25), 3)
    elif product_type == "Jumbo":
        rate = round(np.random.normal(6.875, 0.35), 3)
    else:
        rate = round(np.random.normal(6.25, 0.3), 3)

    fico_score = int(np.random.normal(720, 40))
    fico_score = max(580, min(fico_score, 820))

    cltv = int(np.random.normal(80, 8))
    cltv = max(50, min(cltv, 100))

    dti = int(np.random.normal(36, 7))
    dti = max(20, min(dti, 55))

    lock_term_days = random.choices([15, 30, 45, 60], weights=[10, 45, 30, 15], k=1)[0]

    loan_status = random.choices(
        loan_statuses,
        weights=[70, 10, 8, 7, 5],
        k=1
    )[0]

    if loan_status == "Funded":
        pull_through_flag = 1
        funding_lag = random.randint(20, 60)
        funding_date = application_date + timedelta(days=funding_lag)
    else:
        pull_through_flag = 0
        funding_date = None

    loan_officer_id = random.choice(loan_officers)
    underwriter_id = random.choice(underwriters)
    processor_id = random.choice(processors)
    account_executive_id = random.choice(account_execs) if channel == "Wholesale" and account_execs else None

    lo_row = employees_df.loc[employees_df["employee_id"] == loan_officer_id].iloc[0]
    target_bps = lo_row["target_bps"] if pd.notnull(lo_row["target_bps"]) else 100
    comp_bps = int(max(60, min(target_bps + np.random.randint(-10, 11), 150)))

    comp_amount = round(loan_amount * (comp_bps / 10000.0), 2)

    branch_id = int(lo_row["branch_id"])

    r = random.random()
    if loan_status != "Funded":
        num_extensions = 0
    else:
        if product_type == "Jumbo" or loan_purpose == "Purchase":
            if r < 0.55:
                num_extensions = 0
            elif r < 0.80:
                num_extensions = 1
            elif r < 0.95:
                num_extensions = 2
            else:
                num_extensions = random.choice([3, 4])
        else:
            if r < 0.75:
                num_extensions = 0
            elif r < 0.92:
                num_extensions = 1
            elif r < 0.98:
                num_extensions = 2
            else:
                num_extensions = 3

    loans.append({
        "loan_number": loan_number,
        "application_date": application_date.date(),
        "lock_date": lock_date.date(),
        "funding_date": funding_date.date() if funding_date else "",
        "loan_amount": loan_amount,
        "property_state": property_state,
        "property_type": property_type,
        "occupancy_type": occupancy_type,
        "loan_purpose": loan_purpose,
        "channel": channel,
        "product_type": product_type,
        "rate": rate,
        "fico_score": fico_score,
        "cltv": cltv,
        "dti": dti,
        "lock_term_days": lock_term_days,
        "loan_status": loan_status,
        "loan_officer_id": loan_officer_id,
        "account_executive_id": account_executive_id if account_executive_id is not None else "",
        "underwriter_id": underwriter_id,
        "processor_id": processor_id,
        "comp_bps": comp_bps,
        "comp_amount": comp_amount,
        "branch_id": branch_id,
        "pull_through_flag": pull_through_flag,
        "num_extensions": num_extensions
    })

loans_df = pd.DataFrame(loans)

# Save loans.csv
loans_df.to_csv("loans.csv", index=False)

# -----------------------------
# 3. Generate loan_extensions.csv
# -----------------------------

extension_rows = []
extension_id_counter = 90001

reason_codes = ["Borrower Delay", "Appraisal Delay", "Underwriting Conditions", "Title Issues", "Rate Shopping"]

for _, row in loans_df.iterrows():
    num_ext = int(row["num_extensions"])
    if num_ext <= 0:
        continue

    loan_number = int(row["loan_number"])
    loan_amount = float(row["loan_amount"])
    lock_date = datetime.strptime(str(row["lock_date"]), "%Y-%m-%d")
    lock_term_days = int(row["lock_term_days"])

    original_expiration = lock_date + timedelta(days=lock_term_days)
    current_expiration = original_expiration

    for seq in range(1, num_ext + 1):
        if seq >= 3:
            extension_days = 15
            extension_fee_percent = 0.0025
        else:
            extension_days = random.choices([5, 15], weights=[50, 50], k=1)[0]
            extension_fee_percent = 0.00125 if extension_days == 5 else 0.0025

        extension_fee_amount = round(loan_amount * extension_fee_percent, 2)

        request_date = current_expiration - timedelta(days=random.randint(1, 5))
        lock_exp_before = current_expiration
        lock_exp_after = current_expiration + timedelta(days=extension_days)
        current_expiration = lock_exp_after

        requested_by_role = random.choice(["Loan Officer", "Processor", "Broker"])
        approved_by_employee_id = random.choice(underwriters)

        reason_code = random.choice(reason_codes)

        extension_rows.append({
            "extension_id": extension_id_counter,
            "loan_number": loan_number,
            "extension_sequence": seq,
            "extension_request_date": request_date.date(),
            "extension_days": extension_days,
            "extension_fee_percent": extension_fee_percent,
            "extension_fee_amount": extension_fee_amount,
            "requested_by_role": requested_by_role,
            "approved_by_employee_id": approved_by_employee_id,
            "reason_code": reason_code,
            "lock_expiration_date_before": lock_exp_before.date(),
            "lock_expiration_date_after": lock_exp_after.date()
        })

        extension_id_counter += 1

extensions_df = pd.DataFrame(extension_rows)

# Save loan_extensions.csv
extensions_df.to_csv("loan_extensions.csv", index=False)

print("Generated employees.csv, loans.csv, and loan_extensions.csv")


Generated employees.csv, loans.csv, and loan_extensions.csv
